# Spaghetti
Make spaghetti plots for ENSO events

## Imports

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import pandas as pd

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_spaghetti(idx, data, peak_month, event_type=None, q=0.95, is_warm=True):
    """
    Get hovmoller composite based on specified:
    - data: used to compute index/make composite
    - peak_month: month to center composite on
    - q: quantile threshold for composite
    """

    ## handle warm/cold case
    if is_warm:
        kwargs = dict(q=q, check_cutoff=lambda x, cut: x > cut)
    else:
        kwargs = dict(q=1 - q, check_cutoff=lambda x, cut: x < cut)

    ## kwargs for composite
    kwargs = dict(
        kwargs,
        avg=False,
        peak_month=peak_month,
        idx=idx,
        data=data,
        event_type=event_type,
    )

    ## composite of projected data
    spag = src.utils.make_composite(**kwargs)

    return spag


def save(fig, fname, dpi=800, do_save=False):
    """save figure to file"""

    ## get save directory
    # save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "egu_poster_figs")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    if do_save:
        fig.savefig(fname, dpi=dpi, format="pdf")

    return


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


def load_h_ohc_idxs():
    """load OHC-based h indices"""

    ## specify subsetting funcs
    LONS_E = dict(longitude=slice(210, 270))
    LONS_W = dict(longitude=slice(120, 180))

    ## averaging funcs
    lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
    lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

    ## load spatial data
    _, anom_wide = load_consolidated_wide()

    ## empty array to hold result
    h_idxs = xr.Dataset()

    ## compute ohc
    h_idxs["h_w_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    h_idxs["h_e_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

    h_idxs["h_ohc"] = 0.5 * (h_idxs["h_w_ohc"] + h_idxs["h_e_ohc"])
    h_idxs["dh_ohc"] = h_idxs["h_e_ohc"] - h_idxs["h_w_ohc"]

    return h_idxs

## Load data

In [ ]:
## Th data
Th = src.utils.load_cesm_indices(load_z20=True)

## spatial
_, anom = src.utils.load_consolidated()
Th["sst"] = anom["sst"]

#### get windowed data

In [ ]:
## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

### Early and late

In [ ]:
## specify years
years = [2010, 2080]

## get early and late
Th_early = Th_rolling.sel(year=years[0])
Th_late = Th_rolling.sel(year=years[1])

## remove median
Th_early = Th_early.groupby("time.month") - Th_early.groupby("time.month").median()
Th_late = Th_late.groupby("time.month") - Th_late.groupby("time.month").median()

## compute spaghetti/composite

In [ ]:
## specify composite specs
comp_kwargs = dict(
    is_warm=True,
    event_type=None,
    peak_month=12,
    q=0.75,
)

## specify variable to use for composite
VARNAME = "T_34"

## do the compute
spag_early = get_spaghetti(
    idx=Th_early[VARNAME], data=Th_early, **comp_kwargs
).drop_vars("year")
spag_late = get_spaghetti(idx=Th_late[VARNAME], data=Th_late, **comp_kwargs).drop_vars(
    "year"
)

## switch sign
if not (comp_kwargs["is_warm"]):
    spag_early = -spag_early
    spag_late = -spag_late

In [ ]:
def get_zonal_section(x):
    idx = dict(longitude=slice(140, 280), latitude=slice(-5, 5))
    return x.sel(idx).mean("latitude")


def get_meri_section(x):
    idx = dict(longitude=slice(220, 280), latitude=slice(-15, 15))
    return x.sel(idx).mean("longitude")


def get_fn_wrapper(x, fn):
    return src.utils.reconstruct_wrapper(
        xr.merge([x["sst"], anom["sst_comp"]]),
        fn=fn,
    )["sst"]


spag_early["zonal"] = get_fn_wrapper(spag_early, fn=get_zonal_section)

spag_late["zonal"] = get_fn_wrapper(spag_late, fn=get_zonal_section)

In [ ]:
LAG = 3
sel_ = lambda x: x.sel(lag=slice(LAG, LAG)).mean("lag")

plot_kwargs = dict(c="gray", lw=0.5, alpha=0.5)

fig, axs = plt.subplots(2, 1, figsize=(4, 5), layout="constrained")

## loop thru early/late
for ax, spag_, ls in zip(axs, [spag_early, spag_late], ["-", "--"]):

    ## loop thru samples
    for s in spag_.sample:

        ## plot data
        ax.plot(spag_.longitude, sel_(spag_["zonal"].sel(sample=s)), **plot_kwargs)

    ## mean
    ax.plot(spag_.longitude, sel_(spag_["zonal"].mean("sample")), c="k", lw=2, ls=ls)

    ## guidelines
    ax.axhline(0, ls="--", c="k", lw=1)

axs[1].plot(spag_.longitude, sel_(spag_early["zonal"].mean("sample")), c="k", lw=2)

src.utils.set_lims(axs)
# for a

plt.show()

In [ ]:
LAGS = dict(lag=slice(-3, 3))

import xeofs as xe

eofs_kwargs = dict(n_modes=10, center=True)
eofs_early = xe.single.EOF(**eofs_kwargs)
eofs_late = xe.single.EOF(**eofs_kwargs)

## get early/late
X_early = spag_early["zonal"].sel(LAGS)
X_late = spag_late["zonal"].sel(LAGS)

## fit model
eofs_early.fit(X_early, dim="sample")
eofs_late.fit(X_late, dim="sample")

In [ ]:
## get scaling factor
N_early = len(X_early.sample)
N_late = len(X_late.sample)
fac_early = np.sqrt(eofs_early.explained_variance())
fac_late = np.sqrt(eofs_late.explained_variance())

## check scaling
x0 = eofs_early.scores(normalized=True) * fac_early * np.sqrt(N_early - 1)
x1 = eofs_early.scores(normalized=False)
print(np.allclose(x0, x1))
print(
    np.allclose(
        (eofs_early.components() ** 2).sum(["longitude", "lag"]).values,
        np.ones(eofs_kwargs["n_modes"]),
    )
)

## get scaled components
comps_early = eofs_early.components() * fac_early
comps_late = eofs_late.components() * fac_late

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.bar(
    eofs_early.scores().mode,
    eofs_early.explained_variance_ratio(),
    alpha=0.5,
    label="early",
)

ax.bar(
    eofs_late.scores().mode,
    eofs_late.explained_variance_ratio(),
    fill=None,
    edgecolor="k",
    label="late",
)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
for ax, eofs in zip(axs, [eofs_early, eofs_late]):
    ax.scatter(
        eofs.scores().isel(mode=0),
        eofs.scores().isel(mode=1),
        s=15,
    )
    # ax.set_aspect("equal")
    ax_kwargs = dict(ls="--", c="k", lw=1)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)

src.utils.set_lims(axs)

plt.show()

In [ ]:
sel_ = lambda x: x.sel(lag=slice(1, 3)).mean("lag")
colors = sns.color_palette()

plot_kwargs = dict(c="gray", lw=0.5, alpha=0.5)

fig, ax = plt.subplots(figsize=(4, 3), layout="constrained")
# for ax, comps_ in zip(axs, [comps_early, comps_late]):

for comps_, ls in zip([comps_early, comps_late], ["-", "--"]):
    ax.plot(X_early.longitude, sel_(comps_.sel(mode=1)), c=colors[0], ls=ls)
    ax.plot(X_early.longitude, sel_(comps_.sel(mode=2)), c=colors[1], ls=ls)
ax.axhline(0, ls="--", c="k", lw=1)


plt.show()

do time/longitude EOFs on these?

## plots

### Mean

Clean up; add bounds